# Momentum Quality App MVP

This notebook version is designed for GitHub upload and quick review. For the interactive dashboard, use the Streamlit app files in the repository.

## Install requirements
Run this cell in Colab/Jupyter if needed.

In [ ]:
# Uncomment if packages are missing
# !pip install -r requirements.txt

print("Requirements:")
print("""streamlit==1.41.1
pandas==2.2.3
numpy==2.2.1
""")

## Sample data

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd


def make_price_series(seed: int, start_price: float, trend: float, volatility: float, force_up_days: int = 0):
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=280)
    returns = rng.normal(trend / 252, volatility / np.sqrt(252), size=len(dates))
    prices = start_price * np.exp(np.cumsum(returns))

    if force_up_days > 0:
        base = prices[-force_up_days - 1]
        increments = np.linspace(0.004, 0.025, force_up_days)
        for i in range(force_up_days):
            base *= 1 + increments[i]
            prices[-force_up_days + i] = base

    volumes = rng.integers(500_000, 8_000_000, len(dates))
    if force_up_days > 0:
        volumes[-force_up_days:] = volumes[-force_up_days:] * rng.integers(2, 5)

    return pd.DataFrame({"close": prices, "volume": volumes}, index=dates)


def load_sample_universe():
    benchmark = make_price_series(999, 500, 0.15, 0.12, 0)
    universe = {
        "BB": {
            "company": "BlackBerry",
            "prices": make_price_series(1, 4.2, 1.15, 0.45, 7),
            "fundamentals": {
                "eps": 0.42,
                "previous_eps": 0.18,
                "eps_growth_pct": 133.3,
                "pe_ratio": 26.7,
                "shareholder_equity": 2_100_000_000,
                "previous_shareholder_equity": 1_880_000_000,
                "equity_growth_pct": 11.7,
            },
            "sentiment_score": 0.72,
        },
        "NVDA": {
            "company": "Nvidia",
            "prices": make_price_series(2, 85, 0.75, 0.32, 4),
            "fundamentals": {
                "eps": 2.9,
                "previous_eps": 2.2,
                "eps_growth_pct": 31.8,
                "pe_ratio": 38.0,
                "shareholder_equity": 55_000_000_000,
                "previous_shareholder_equity": 48_000_000_000,
                "equity_growth_pct": 14.6,
            },
            "sentiment_score": 0.55,
        },
        "XYZ": {
            "company": "Example Recovery Plc",
            "prices": make_price_series(3, 20, -0.25, 0.55, 6),
            "fundamentals": {
                "eps": -0.25,
                "previous_eps": -0.40,
                "eps_growth_pct": 37.5,
                "pe_ratio": -12.0,
                "shareholder_equity": -500_000_000,
                "previous_shareholder_equity": -450_000_000,
                "equity_growth_pct": -11.1,
            },
            "sentiment_score": -0.35,
        },
        "ABC": {
            "company": "Example Quality Corp",
            "prices": make_price_series(4, 35, 0.32, 0.20, 3),
            "fundamentals": {
                "eps": 1.85,
                "previous_eps": 1.65,
                "eps_growth_pct": 12.1,
                "pe_ratio": 18.5,
                "shareholder_equity": 4_200_000_000,
                "previous_shareholder_equity": 3_900_000_000,
                "equity_growth_pct": 7.7,
            },
            "sentiment_score": 0.22,
        },
        "HOT": {
            "company": "Example Hot Momentum Inc",
            "prices": make_price_series(5, 9, 0.95, 0.75, 10),
            "fundamentals": {
                "eps": 0.05,
                "previous_eps": -0.05,
                "eps_growth_pct": 200.0,
                "pe_ratio": 180.0,
                "shareholder_equity": 120_000_000,
                "previous_shareholder_equity": 80_000_000,
                "equity_growth_pct": 50.0,
            },
            "sentiment_score": 0.85,
        },
    }
    return universe, benchmark


## Scoring engine

In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Dict, Optional

import numpy as np
import pandas as pd


LOOKBACKS = {
    "5D": 5,
    "1W": 5,
    "2W": 10,
    "3W": 15,
    "1M": 21,
    "3M": 63,
    "6M": 126,
    "9M": 189,
    "1Y": 252,
}


def pct_return(series: pd.Series, periods: int) -> float:
    if len(series) <= periods or series.iloc[-periods - 1] == 0:
        return np.nan
    return (series.iloc[-1] / series.iloc[-periods - 1] - 1) * 100


def consecutive_up_days(close: pd.Series, max_days: int = 10) -> int:
    diffs = close.diff().dropna()
    count = 0
    for value in reversed(diffs.tail(max_days).tolist()):
        if value > 0:
            count += 1
        else:
            break
    return count


def rsi(close: pd.Series, period: int = 14) -> float:
    if len(close) < period + 1:
        return np.nan
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(period).mean()
    loss = -delta.clip(upper=0).rolling(period).mean()
    rs = gain / loss.replace(0, np.nan)
    value = 100 - (100 / (1 + rs.iloc[-1]))
    if np.isnan(value):
        return 100.0 if gain.iloc[-1] > 0 else np.nan
    return float(value)


def win_rate(close: pd.Series, periods: int = 252) -> float:
    changes = close.diff().dropna().tail(periods)
    if len(changes) == 0:
        return np.nan
    return (changes.gt(0).sum() / len(changes)) * 100


def normalize(value: float, low: float, high: float, inverse: bool = False) -> float:
    if value is None or np.isnan(value):
        return 0.0
    if high == low:
        return 0.0
    score = (value - low) / (high - low) * 100
    score = max(0.0, min(100.0, score))
    return 100 - score if inverse else score


def score_row(row: Dict) -> Dict:
    """Create component scores and final Momentum Quality Score."""
    up_days_score = normalize(row.get("consecutive_up_days", 0), 0, 10)
    returns_score = np.nanmean([
        normalize(row.get("return_5D", np.nan), -10, 20),
        normalize(row.get("return_1M", np.nan), -20, 60),
        normalize(row.get("return_3M", np.nan), -30, 100),
        normalize(row.get("return_1Y", np.nan), -50, 200),
    ])
    price_momentum = np.nanmean([up_days_score, returns_score])

    relative_volume_score = normalize(row.get("relative_volume", np.nan), 0.5, 5.0)
    high_gap_score = normalize(row.get("gap_to_52w_high_pct", np.nan), 0, 15, inverse=True)

    rsi_value = row.get("rsi", np.nan)
    if np.isnan(rsi_value):
        rsi_score = 0
    elif 55 <= rsi_value <= 75:
        rsi_score = 100
    elif 45 <= rsi_value < 55:
        rsi_score = 65
    elif 75 < rsi_value <= 85:
        rsi_score = 70
    elif rsi_value > 85:
        rsi_score = 45
    else:
        rsi_score = 25

    eps_score = 0
    if row.get("eps", np.nan) > 0:
        eps_score += 45
    if row.get("eps_growth_pct", np.nan) > 0:
        eps_score += 55

    pe = row.get("pe_ratio", np.nan)
    if np.isnan(pe) or pe <= 0:
        pe_score = 0
    elif pe <= 40:
        pe_score = 100
    elif pe <= 80:
        pe_score = 65
    else:
        pe_score = 35

    fundamentals = np.nanmean([eps_score, pe_score])

    equity_score = 0
    if row.get("shareholder_equity", np.nan) > 0:
        equity_score += 50
    if row.get("equity_growth_pct", np.nan) > 0:
        equity_score += 50

    alpha_score = normalize(row.get("alpha_252D", np.nan), -50, 100)
    winrate_score = normalize(row.get("win_rate_252D", np.nan), 40, 70)
    alpha_quality = np.nanmean([alpha_score, winrate_score])

    sentiment_score = normalize(row.get("sentiment_score", 0), -1, 1)

    final = (
        price_momentum * 0.20
        + relative_volume_score * 0.15
        + high_gap_score * 0.10
        + rsi_score * 0.10
        + fundamentals * 0.15
        + equity_score * 0.10
        + alpha_quality * 0.10
        + sentiment_score * 0.10
    )

    return {
        "price_momentum_score": round(price_momentum, 1),
        "volume_score": round(relative_volume_score, 1),
        "high_gap_score": round(high_gap_score, 1),
        "rsi_score": round(rsi_score, 1),
        "fundamental_score": round(fundamentals, 1),
        "equity_score": round(equity_score, 1),
        "alpha_quality_score": round(alpha_quality, 1),
        "sentiment_component_score": round(sentiment_score, 1),
        "momentum_quality_score": round(final, 1),
    }


def compute_metrics(
    symbol: str,
    company: str,
    prices: pd.DataFrame,
    benchmark_prices: pd.DataFrame,
    fundamentals: Dict,
    sentiment_score: float = 0.0,
) -> Dict:
    df = prices.copy().sort_index()
    close = df["close"]
    volume = df["volume"]
    benchmark_close = benchmark_prices["close"].sort_index()

    latest_price = float(close.iloc[-1])
    high_52w = float(close.tail(252).max())
    gap = ((high_52w - latest_price) / high_52w) * 100 if high_52w else np.nan
    rel_vol = float(volume.iloc[-1] / volume.tail(30).mean()) if len(volume) >= 30 else np.nan

    row = {
        "symbol": symbol,
        "company": company,
        "latest_price": latest_price,
        "high_52w": high_52w,
        "gap_to_52w_high_pct": gap,
        "latest_volume": int(volume.iloc[-1]),
        "avg_volume_30D": int(volume.tail(30).mean()),
        "relative_volume": rel_vol,
        "consecutive_up_days": consecutive_up_days(close),
        "rsi": rsi(close),
        "win_rate_252D": win_rate(close, 252),
        "sentiment_score": sentiment_score,
        **fundamentals,
    }

    for label, days in LOOKBACKS.items():
        row[f"return_{label}"] = pct_return(close, days)
        stock_return = row[f"return_{label}"]
        bench_return = pct_return(benchmark_close, days)
        row[f"benchmark_return_{label}"] = bench_return
        row[f"alpha_{label}"] = stock_return - bench_return if not np.isnan(stock_return) and not np.isnan(bench_return) else np.nan

    row.update(score_row(row))
    return row


## Run scanner in notebook

In [ ]:

# Load sample stock data and calculate scores
prices, fundamentals, sentiment = build_sample_dataset()
results = score_universe(prices, fundamentals, sentiment)

# Example filters similar to the app sidebar
filtered = results[
    (results["consecutive_up_days"] >= 2) &
    (results["gap_to_52w_high_pct"] <= 20) &
    (results["relative_volume"] >= 1.0) &
    (results["eps_positive"] == True) &
    (results["pe_positive"] == True)
].sort_values("total_score", ascending=False)

filtered


## Streamlit app source
This is the dashboard version. Save as `app.py` and run `streamlit run app.py`.

In [ ]:
from __future__ import annotations

import pandas as pd
import streamlit as st

from sample_data import load_sample_universe
from scoring import compute_metrics

st.set_page_config(page_title="Momentum Quality Scanner", layout="wide")

st.title("Momentum Quality Scanner")
st.caption("Find shares with technical momentum, strong volume, improving fundamentals, positive alpha, and supportive sentiment.")

with st.sidebar:
    st.header("User Parameters")
    min_up_days = st.slider("Minimum consecutive up days", 0, 10, 3)
    max_gap = st.slider("Maximum gap from 52-week high (%)", 0.0, 100.0, 5.0, 0.1)
    min_rel_volume = st.slider("Minimum relative volume", 0.0, 10.0, 1.5, 0.1)
    require_positive_eps = st.checkbox("Require positive EPS", value=True)
    require_rising_eps = st.checkbox("Require rising EPS", value=True)
    require_positive_pe = st.checkbox("Require positive P/E", value=True)
    require_positive_equity = st.checkbox("Require positive shareholder equity", value=True)
    require_rising_equity = st.checkbox("Require rising shareholder equity", value=False)
    min_alpha = st.slider("Minimum 1-year alpha vs benchmark (%)", -100.0, 200.0, 0.0, 1.0)
    min_score = st.slider("Minimum Momentum Quality Score", 0, 100, 60)
    rsi_min, rsi_max = st.slider("Acceptable RSI range", 0, 100, (45, 85))

universe, benchmark = load_sample_universe()
rows = []
for symbol, data in universe.items():
    rows.append(
        compute_metrics(
            symbol=symbol,
            company=data["company"],
            prices=data["prices"],
            benchmark_prices=benchmark,
            fundamentals=data["fundamentals"],
            sentiment_score=data["sentiment_score"],
        )
    )

results = pd.DataFrame(rows)

filtered = results.copy()
filtered = filtered[filtered["consecutive_up_days"] >= min_up_days]
filtered = filtered[filtered["gap_to_52w_high_pct"] <= max_gap]
filtered = filtered[filtered["relative_volume"] >= min_rel_volume]
filtered = filtered[filtered["alpha_1Y"] >= min_alpha]
filtered = filtered[filtered["momentum_quality_score"] >= min_score]
filtered = filtered[(filtered["rsi"] >= rsi_min) & (filtered["rsi"] <= rsi_max)]

if require_positive_eps:
    filtered = filtered[filtered["eps"] > 0]
if require_rising_eps:
    filtered = filtered[filtered["eps_growth_pct"] > 0]
if require_positive_pe:
    filtered = filtered[filtered["pe_ratio"] > 0]
if require_positive_equity:
    filtered = filtered[filtered["shareholder_equity"] > 0]
if require_rising_equity:
    filtered = filtered[filtered["equity_growth_pct"] > 0]

filtered = filtered.sort_values("momentum_quality_score", ascending=False)

summary_cols = [
    "symbol", "company", "momentum_quality_score", "consecutive_up_days",
    "latest_price", "high_52w", "gap_to_52w_high_pct", "relative_volume",
    "rsi", "eps", "eps_growth_pct", "pe_ratio", "equity_growth_pct",
    "return_5D", "return_1M", "return_3M", "return_1Y", "alpha_1Y",
    "win_rate_252D", "sentiment_score"
]

st.subheader("Ranked Matches")
st.dataframe(
    filtered[summary_cols].style.format({
        "momentum_quality_score": "{:.1f}",
        "latest_price": "{:.2f}",
        "high_52w": "{:.2f}",
        "gap_to_52w_high_pct": "{:.2f}%",
        "relative_volume": "{:.2f}x",
        "rsi": "{:.1f}",
        "eps": "{:.2f}",
        "eps_growth_pct": "{:.1f}%",
        "pe_ratio": "{:.1f}",
        "equity_growth_pct": "{:.1f}%",
        "return_5D": "{:.1f}%",
        "return_1M": "{:.1f}%",
        "return_3M": "{:.1f}%",
        "return_1Y": "{:.1f}%",
        "alpha_1Y": "{:.1f}%",
        "win_rate_252D": "{:.1f}%",
        "sentiment_score": "{:.2f}",
    }),
    use_container_width=True,
    hide_index=True,
)

st.subheader("All Stocks Scored")
st.dataframe(results[summary_cols].sort_values("momentum_quality_score", ascending=False), use_container_width=True, hide_index=True)

st.subheader("Score Breakdown")
selected = st.selectbox("Select a stock", results["symbol"].tolist())
row = results[results["symbol"] == selected].iloc[0]
score_cols = [
    "price_momentum_score", "volume_score", "high_gap_score", "rsi_score",
    "fundamental_score", "equity_score", "alpha_quality_score", "sentiment_component_score"
]
st.bar_chart(row[score_cols])

st.markdown("""
### How to connect real data
This MVP currently runs with sample data so you can test the product immediately.
Replace `sample_data.py` with connectors for:

- Market prices and volume: Polygon, Tiingo, Alpha Vantage, Twelve Data, Yahoo Finance-style feeds
- Fundamentals: Financial Modeling Prep, Intrinio, EODHD, IEX Cloud, SEC/company filings
- News sentiment: Benzinga, Finnhub, NewsAPI, RavenPack-style sentiment, or your own AI classifier
- Benchmark: S&P 500, Nasdaq, FTSE 100, TSX, or user-selected index

### Important
This app is a decision-support screener. It should show users which shares match their selected rules, not guarantee profit or provide personal financial advice.
""")


## README

# Momentum Quality Scanner MVP

A fast MVP for screening shares using:

- Consecutive up days
- Relative volume
- 52-week high proximity
- RSI
- EPS positivity and EPS growth
- Positive P/E
- Positive and rising shareholder equity / net assets
- Multi-timeframe returns
- Alpha vs benchmark
- Win rate
- News sentiment score

## Run locally

```bash
python -m venv .venv
source .venv/bin/activate   # Windows: .venv\Scripts\activate
pip install -r requirements.txt
streamlit run app.py
```

## Files

- `app.py` — Streamlit user interface
- `scoring.py` — all calculations and scoring logic
- `sample_data.py` — realistic sample stock universe
- `requirements.txt` — dependencies

## Next development step

Replace `sample_data.py` with real data connectors:

1. Daily prices and volume
2. Fundamentals: EPS, P/E, shareholder equity
3. Benchmark index prices
4. News and sentiment

## Suggested production architecture

- Frontend: React / Next.js or Streamlit for early users
- Backend: FastAPI
- Database: PostgreSQL
- Jobs: Cron, Celery, or Airflow
- Hosting: Render, AWS, Railway, or Azure
- Alerts: email, Telegram, WhatsApp, or push notifications

## Disclaimer

This tool is for screening and research only. It is not financial advice.
